## Generate TEEHR crosswalk data from NWM routelink files for all domains

In [ ]:
import xarray as xr
import pandas as pd
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime

import teehr

## Routelink Dates and Info:

#### CONUS:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.17/parm/domain/Routelink_CONUS.nc    
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_CONUS.nc   
Created 4/12/2019 by NCAR, contains 2776738 feature IDs  
Corresponds to NWM v2.1/2.2, 3.0

Prior version:  Routelink_CONUS_v2.0.nc (no longer available online, stored in TEEHR data warehouse on S3)   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_CONUS_v2.0.nc   
Created 11/7/2018 by NCAR, contains 2729077 feature IDs   
Corresponds to NWM v1.2, 2.0   

#### Hawaii:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_hawaii/RouteLink_HI.nc   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_HI.nc   
Created 1/12/2018 by NCAR, contains 13637 feature IDs   
Corresponds to NWM v2.0, 2.1/2.2, 3.0  (Hawaii was added in v2.0)   

#### Puerto Rico:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_puertorico/RouteLink_PRVI.nc   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_PRVI.nc   
Created 5/27/2021 by NCAR, contains 14017 feature IDs   
Corresponds to NWM v2.1/2.2, 3.0  (Puerto Rico was added in v2.1)   

#### Alaska:

Current version: https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_alaska/RouteLink_AK.nc   
Read from S3: s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_AK.nc   
Created 4/21/2022 by NCAR, contains 391528 feature IDs   
Corresponds to NWM 3.0  (Alaska was added in v3.0)   

#### NWM version dates for reference:
**v1.2**----------2018-09-17 – 2019-06-18    
**v2.0**----------2019-06-19 – 2021-04-19           
**v2.1/v2.2**----2021-04-20 – 2023-09-18    
**v3.0**----------2023-09-19 – present        

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")
def extract_nwm_crosswalk_from_routelink(routelink_file, nwm_prefix):

    print(f"processing {routelink_file}")

    # Open the NetCDF file
    ds = xr.open_dataset(routelink_file, engine="h5netcdf", chunks={})

    # extract the links and gages
    df = ds[['link', 'gages']].to_dataframe().reset_index(drop=True)[['link','gages']]

    # remove blanks
    df['gages'] = df['gages'].str.decode('utf-8').str.strip()
    crosswalk = df[df['gages'] != ""].copy().reset_index(drop=True)

    # format into a teehr crosswalk table
    crosswalk['gages'] = crosswalk['gages'].str.zfill(8).radd('usgs-')
    crosswalk['link'] = crosswalk['link'].astype(str).radd(nwm_prefix + '-')
    crosswalk = crosswalk.rename(columns={'link': 'secondary_location_id', 'gages': 'primary_location_id'})
    properties_dict = {
        "source" : "NWS",
        "url" : routelink_file,
        "date_retreived" : fetch_date,
    }
    crosswalk['properties'] = [properties_dict] * len(crosswalk)

    return crosswalk

### Read routelink files and convert to TEEHR format

In [ ]:
routelink_files = {
    "conus_30":"s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_CONUS.nc",
    "conus_20":"s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_CONUS_v2.0.nc",
    "hawaii":"s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_HI.nc",
    "puertorico":"s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_PRVI.nc",
    "alaska":"s3://ciroh-rti-public-data/teehr-data-warehouse/common/crosswalks/routelinks/RouteLink_AK.nc",
}

# process all routelinks
crosswalk_dict = {}
for domain in ['conus_30','hawaii','puertorico','alaska']:
    crosswalk_dict[domain] = extract_nwm_crosswalk_from_routelink(routelink_files[domain], 'nwm30')

crosswalk_dict['conus_20'] = extract_nwm_crosswalk_from_routelink(routelink_files['conus_20'], 'nwm20')

### Build crosswalk tables for each NWM version

In [ ]:
# v3.0 - conus_30, hawaii, puertorico, alaska
cross_list = []
for domain in ['conus_30','hawaii','puertorico','alaska']:
    cross_list.append(crosswalk_dict[domain])
nwm30_crosswalk = pd.concat(cross_list, ignore_index=True)
display(nwm30_crosswalk)

# v2.1/2.2 - conus_30, hawaii, puertorico
cross_list = []
for domain in ['conus_30','hawaii','puertorico']:
    cross_list.append(crosswalk_dict[domain])
nwm22_crosswalk = pd.concat(cross_list, ignore_index=True)
nwm22_crosswalk['secondary_location_id'] = nwm22_crosswalk['secondary_location_id'].str.replace('nwm30','nwm22')
display(nwm22_crosswalk)
nwm21_crosswalk = nwm22_crosswalk.copy()
nwm21_crosswalk['secondary_location_id'] = nwm21_crosswalk['secondary_location_id'].str.replace('nwm22','nwm21')
display(nwm21_crosswalk)

# v2.0 - conus_20, hawaii
cross_list = []
for domain in ['conus_20','hawaii']:
    cross_list.append(crosswalk_dict[domain])
nwm20_crosswalk = pd.concat(cross_list, ignore_index=True)
nwm20_crosswalk['secondary_location_id'] = nwm20_crosswalk['secondary_location_id'].str.replace('nwm30','nwm20')
display(nwm20_crosswalk)

# v1.2 - conus_20
nwm12_crosswalk = crosswalk_dict['conus_20']
nwm12_crosswalk['secondary_location_id'] = nwm12_crosswalk['secondary_location_id'].str.replace('nwm20','nwm12')
display(nwm12_crosswalk)

### Filter out crosswalk primary_ids that are not in our updated locations table

In [ ]:
missing_df = pd.read_csv("/data/common/crosswalks/routelinks/missing_routelink_primary_ids.csv")
print(missing_df.index.size)
missing_df.head()

In [ ]:
before_12 = nwm12_crosswalk.index.size
after_12 = nwm12_crosswalk[~nwm12_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])].index.size
before_20 = nwm20_crosswalk.index.size
after_20 = nwm20_crosswalk[~nwm20_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])].index.size
before_21 = nwm21_crosswalk.index.size
after_21 = nwm21_crosswalk[~nwm21_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])].index.size
before_30 = nwm30_crosswalk.index.size
after_30 = nwm30_crosswalk[~nwm30_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])].index.size

In [ ]:
print(f"nwm12 before filter: {before_12}, after filter: {after_12}")
print(f"nwm20 before filter: {before_20}, after filter: {after_20}")
print(f"nwm21 before filter: {before_21}, after filter: {after_21}")
print(f"nwm30 before filter: {before_30}, after filter: {after_30}")

In [ ]:
nwm12_crosswalk = nwm12_crosswalk[~nwm12_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])]
nwm20_crosswalk = nwm20_crosswalk[~nwm20_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])]
nwm21_crosswalk = nwm21_crosswalk[~nwm21_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])]
nwm30_crosswalk = nwm30_crosswalk[~nwm30_crosswalk["primary_location_id"].isin(missing_df["primary_location_id"])]

### Write crosswalks to disk for loading 

In [ ]:
CROSSWALK_DIR = Path("/data/common/crosswalks/routelinks/updated_crosswalks_filtered/")

In [ ]:
nwm12_crosswalk.to_parquet(Path(CROSSWALK_DIR, "nwm12_crosswalk.parquet"), index=False)
nwm20_crosswalk.to_parquet(Path(CROSSWALK_DIR, "nwm20_crosswalk.parquet"), index=False)
nwm21_crosswalk.to_parquet(Path(CROSSWALK_DIR, "nwm21_crosswalk.parquet"), index=False)
nwm30_crosswalk.to_parquet(Path(CROSSWALK_DIR, "nwm30_crosswalk.parquet"), index=False)

### Load to TEEHR

In [ ]:
pd.read_parquet(Path(CROSSWALK_DIR, "nwm12_crosswalk.parquet"))["properties"][0]

In [ ]:
from teehr.evaluation.evaluation import create_spark_session

spark = create_spark_session(
    aws_profile="admin-user"
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

#### Check to see how many primary IDs are in the crosswalks that are not in our locations table

In [ ]:
sdf = ev.spark.read.parquet("/data/common/crosswalks/routelinks/updated_crosswalks_filtered/*.parquet")
sdf.show(n=6, truncate=False)

In [ ]:
sdf.count()

In [ ]:
xwalk_df = sdf.select("primary_location_id").distinct().toPandas()
xwalk_df.head()

In [ ]:
locations_df = ev.locations.to_pandas()

In [ ]:
mask = xwalk_df["primary_location_id"].isin(locations_df["id"])

In [ ]:
missing_xwalk_primary_ids_df = xwalk_df[~mask]  # this should be empty

In [ ]:
missing_xwalk_primary_ids_df

In [ ]:
# missing_xwalk_primary_ids_df.to_csv("/data/common/crosswalks/routelinks/missing_routelink_primary_ids.csv", index=False)

#### Load crosswalks to the warehouse

In [ ]:
%%time
ev.location_crosswalks.load_parquet(
    in_path=CROSSWALK_DIR,
    write_mode="upsert"
)